# 00 — Present Last-FM (`lastfm-u25`) với cô Châu (14–15/11/2026)

Bản trình bày trên notebook cho kết quả Last-FM. Cùng khuôn với `00_present_2026-10-31.ipynb`; mỗi phần ghi người trình bày
**(đề xuất, chưa chốt với cô và Phôn)**. Tài liệu học đi kèm: `E:\KLTN HCMUS\02-lastfm-kaggle\tai-lieu-present-lastfm-u25\` (08, 09, 10).

**Quy tắc của notebook**
- **Không huấn luyện lại.** Chỉ đọc kết quả đã lưu và vẽ hình từ `docs/figures/` hoặc từ dữ liệu đã xử lý.
- **Không gõ số bằng tay.** Mọi con số trong lời dẫn được in ra từ code: từ `results/registry.csv` (qua
  `tkgat.registry.load()`), `results/rq3_models.csv`, `results/rq3_pairs.csv`, `results/significance.csv`,
  `docs/data-report-lastfm-numbers.md` (do `scripts/07_report_numbers_lastfm.py` sinh), hoặc tính trực tiếp từ
  `data/processed/lastfm-u25/` bằng hàm của repo (`tkgat.analysis.eda`). Đoạn trích từ file quyết định (0012, 0014) được in nguyên văn.
- **Chỉ `lastfm-u25`** (25% user của Last-FM, quyết định 0013). Mọi bảng ghi rõ tên run.
- **Điều kiện chạy:** đã dựng `data/processed/lastfm-u25/` (scripts 02, 06); "Restart and run all" không lỗi.

| # | Phần | Người trình bày (đề xuất) |
|---|---|---|
| 1 | Vì sao Last-FM; protocol không đổi; ba câu hỏi nghiên cứu | Phôn |
| 2 | Dữ liệu: nguồn, chia, thời gian, KG, hạn chế của tập cặp, bằng chứng không rò rỉ | Triết |
| 3 | Huấn luyện: quy trình, siêu tham số, tính toán và Kaggle | Phôn (baseline, quy trình) + Triết (KGAT, T-KGAT) |
| 4 | Kết quả và nhận xét | Phôn (baseline) + Triết (KGAT, T-KGAT) |
| 5 | Minh bạch và hạn chế | Triết |
| 6 | Kế hoạch và các câu hỏi xin ý kiến cô | Cả hai |

In [ ]:
import dataclasses
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

from tkgat import registry
from tkgat.analysis import eda
from tkgat.analysis.report import parse_numbers, vn_p, vn_pm
from tkgat.data.leakage import check_split
from tkgat.pipeline.build import load_config, load_saved

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DS = "lastfm-u25"
NUMS = parse_numbers((ROOT / "docs/data-report-lastfm-numbers.md").read_text(encoding="utf-8"))
ML = parse_numbers((ROOT / "docs/data-report-numbers.md").read_text(encoding="utf-8"))
REG_ALL = registry.load(ROOT / "results/registry.csv")  # newest run per dataset/split/model/config/seed/phase
REG = REG_ALL[REG_ALL["dataset"] == DS]
HAVE_DATA = (ROOT / "data/processed" / DS / "meta.json").exists()
NOT_RUN = "chưa chạy"


def V(key):
    # a number or text of docs/data-report-lastfm-numbers.md, exactly as the script wrote it (Vietnamese format)
    if key not in NUMS:
        raise KeyError(f"{key!r} is not in docs/data-report-lastfm-numbers.md")
    return NUMS[key][0]


def M(key):
    # same, for the MovieLens report docs/data-report-numbers.md
    if key not in ML:
        raise KeyError(f"{key!r} is not in docs/data-report-numbers.md")
    return ML[key][0]


def say(text):
    display(Markdown(text))


def show(df):
    # table without the row index; falls back to a plain table when the Styler (jinja2) is missing
    try:
        display(df.style.hide(axis="index"))
    except Exception:
        display(df)


def figure(name, width=760):
    path = ROOT / "docs/figures" / name
    if path.exists():
        display(Image(filename=path, width=width))
    else:
        say(f"*chưa có: `docs/figures/{name}`.*")


def read_text(rel):
    return (ROOT / rel).read_text(encoding="utf-8")


def num(value, digits=4):
    return f"{value:.{digits}f}".replace(".", ",")


def pct(value, digits=1):
    return f"{100 * value:.{digits}f}".replace(".", ",") + "%"


say(
    f"> Số liệu đọc từ `docs/data-report-lastfm-numbers.md` ({len(NUMS)} khóa) và `results/registry.csv`: "
    f"**{len(REG)}** dòng mới nhất mỗi khóa của `{DS}`, commit {', '.join(sorted(REG['git_commit'].unique()))}, "
    f"**{int(REG['git_commit'].str.contains('dirty').sum())}** dòng `-dirty`. "
    f"Freebase là **bản chụp ngày {V('conv.freebase_snapshot')}**, không phải tri thức có sẵn năm 2014: "
    "mọi kết quả dùng KG phải nêu điều này. **Kết quả KGAT và T-KGAT là sơ bộ** (chưa có tag `v0.2-kgat`)."
)
if not HAVE_DATA:
    say('**Chưa có `data/processed/lastfm-u25/`:** các ô đọc dữ liệu đã xử lý sẽ báo "chưa có".')

## 1. Vì sao Last-FM; protocol không đổi; ba câu hỏi nghiên cứu
*Người trình bày: Phôn (đề xuất)*

Theo hướng cô gợi ý (ngày 01/11): chuyển sang Last-FM và lặp lại các thí nghiệm của MovieLens. **Protocol không đổi**: cùng mã chia,
cùng phân vị của t1/t2, k-core chỉ trên train, cùng `check_split`, cùng luật KG, cùng lưới, cùng 5 seed. Chỉ dataset khác.
Dưới đây là phần Context của quyết định 0012, nguyên văn (bao gồm việc quyết định được đưa ra sau khi đã xem test MovieLens):

In [ ]:
decision_0012 = read_text("docs/decisions/0012-lastfm-dataset.md")
ctx = re.search(r"- Context:(.*?)\n- Decision:", decision_0012, re.S)
say(
    "*(nguyên văn từ file quyết định, tiếng Anh)*\n\n> " + " ".join(ctx.group(1).split())
    if ctx
    else "*chưa có: không tìm thấy mục Context trong 0012.*"
)

rows = []
for path in sorted((ROOT / "docs/decisions").glob("001[234]-*.md")):
    lines = path.read_text(encoding="utf-8").splitlines()
    status = next((x for x in lines if x.startswith("- Status:")), "- Status: ?")
    rows.append(
        {
            "quyết định": path.stem,
            "tiêu đề": lines[0].lstrip("# ").strip(),
            "Status (nguyên văn)": status.removeprefix("- Status:").strip(),
        }
    )
pd.set_option("display.max_colwidth", None)
show(pd.DataFrame(rows))

**Ba câu hỏi nghiên cứu** (đọc từ `docs/plan.md`):

In [ ]:
rq = [
    cell.strip()
    for line in read_text("docs/plan.md").splitlines()
    if line.startswith("| RQ")
    for cell in [line.split("|")[1]]
]
say("\n".join(f"- {x}" for x in rq) if rq else "*chưa có: không tìm thấy dòng RQ trong `docs/plan.md`.*")
say(
    "*Câu RQ1 trong `docs/plan.md` viết cho MovieLens. Với Last-FM, phần 2 của notebook là phần tương ứng (đặc điểm "
    "thời gian của dữ liệu và KG). "
    "RQ2 và RQ3 được lặp lại nguyên văn ở phần 4.*"
)

## 2. Dữ liệu Last-FM: nguồn, chia theo thời gian, KG
*Người trình bày: Triết (đề xuất)*

**Ba nguồn ghép lại** (quyết định 0012): (1) tập cặp (user, item) và KG mà tác giả KGAT công bố, không có timestamp; (2) **LFM-1b**,
nguồn timestamp, mỗi cặp giữ **lần nghe sớm nhất**; (3) **bản chụp Freebase** cho ngày của fact, qua ngày tồn tại của đích.
Last-FM không có rating: mọi cặp là tương tác dương (`min_rating` = 1). Không cần liên kết thực thể như MovieLens vì dùng
đúng tập user/item của tác giả.

**Chia theo mốc t1/t2** (phân vị 0,8 và 0,9 của mọi timestamp; 5-core chỉ trên train; val/test chỉ giữ user và item đã có trong train).
Cột `lastfm-u25` là tập của các lần chạy chính thức (25% user, quyết định 0013); cột `lastfm` là bộ đầy đủ, chỉ để đối chiếu (không chạy được trên máy hiện có); ML-100K để so với MovieLens:

In [ ]:
def col(prefix):
    return {
        "cặp (user, item)": V(f"{prefix}.pairs"),
        "t1 / t2": f"{V(f'{prefix}.t1')} / {V(f'{prefix}.t2')}",
        "positive train": V(f"{prefix}.train_pos"),
        "user / item train": f"{V(f'{prefix}.users')} / {V(f'{prefix}.items')}",
        "positive val (user)": f"{V(f'{prefix}.val_pos')} ({V(f'{prefix}.val_users')})",
        "positive test (user)": f"{V(f'{prefix}.test_pos')} ({V(f'{prefix}.test_users')})",
        "val giữ lại": V(f"{prefix}.val_kept_share"),
        "test giữ lại": V(f"{prefix}.test_kept_share"),
        "val mất vì cold user / cold item": f"{V(f'{prefix}.val_dropped_cold_user_share')} / "
        f"{V(f'{prefix}.val_dropped_cold_item_share')}",
        "test mất vì cold user / cold item": f"{V(f'{prefix}.test_dropped_cold_user_share')} / "
        f"{V(f'{prefix}.test_dropped_cold_item_share')}",
        "positive bị 5-core loại khỏi train": V(f"{prefix}.kcore_removed"),
    }


table = pd.DataFrame(
    {
        "lastfm-u25 (chính thức)": col("lastfm-u25"),
        "lastfm (đầy đủ, đối chiếu)": col("lastfm"),
    }
)
display(table)
say(
    f"**ML-100K để so sánh:** giữ lại {V('cmp.ml100k.val_kept_share')} positive val và "
    f"{V('cmp.ml100k.test_kept_share')} test, "
    f"còn {V('cmp.ml100k.val_users')} và {V('cmp.ml100k.test_users')} user; ML-1M: {V('cmp.ml1m.test_kept_share')} "
    f"test, {V('cmp.ml1m.test_users')} user. "
    f"`{DS}` có {V('lastfm-u25.val_users')} user val và {V('lastfm-u25.test_users')} user test, nên kiểm định có sức "
    "mạnh thật."
)
say(
    f"**Cái giá của 25% user:** {V('lastfm-u25.val_dropped_cold_item_share')} positive val và "
    f"{V('lastfm-u25.test_dropped_cold_item_share')} positive test "
    f"bị loại vì **cold item** (bộ đầy đủ: {V('lastfm.val_dropped_cold_item_share')} và "
    f"{V('lastfm.test_dropped_cold_item_share')}). "
    f"Kết quả chỉ nói cho mẫu {V('lastfm-u25.user_fraction')} user ({V('lastfm-u25.users_sampled')}, seed "
    f"{V('lastfm-u25.user_seed')}), không nói cho toàn Last-FM."
)

**Độ cũ Δt của cạnh train trên Last-FM** (điều kiện để Stage 1 có tác dụng). Ba số đầu đọc từ `docs/data-report-lastfm-numbers.md`
(`scripts/07_report_numbers_lastfm.py` tính chúng từ `train.csv` bằng `eda.node_time_spread`, cùng hàm tạo các số của MovieLens, mốc t1)
và so với số của báo cáo dữ liệu MovieLens; các số Δt_norm bên dưới vẫn tính trực tiếp từ `train.csv`:

In [ ]:
if HAVE_DATA:
    meta = json.loads(read_text(f"data/processed/{DS}/meta.json"))
    tr = pd.read_csv(ROOT / f"data/processed/{DS}/train.csv")
    t1 = int(meta["t1"])
    rows = [
        {
            "bộ dữ liệu": "Last-FM u25 (báo cáo dữ liệu Last-FM)",
            "user có mọi cạnh train trong < 1 giờ": V(f"{DS}.spread.user_under_1h"),
            "trung vị độ lệch chuẩn thời gian theo user": f"{V(f'{DS}.spread.user_std_days_median')} ngày",
            "trung vị độ lệch chuẩn thời gian theo item": f"{V(f'{DS}.spread.item_std_days_median')} ngày",
        },
        {
            "bộ dữ liệu": "ML-100K (báo cáo dữ liệu)",
            "user có mọi cạnh train trong < 1 giờ": M("ml100k.user_spread_under_1h"),
            "trung vị độ lệch chuẩn thời gian theo user": f"{M('ml100k.user_std_minutes_median')} phút",
            "trung vị độ lệch chuẩn thời gian theo item": f"{M('ml100k.item_std_days_median')} ngày",
        },
        {
            "bộ dữ liệu": "ML-1M (báo cáo dữ liệu)",
            "user có mọi cạnh train trong < 1 giờ": M("ml1m.user_spread_under_1h"),
            "trung vị độ lệch chuẩn thời gian theo user": f"{M('ml1m.user_std_minutes_median')} phút",
            "trung vị độ lệch chuẩn thời gian theo item": f"{M('ml1m.item_std_days_median')} ngày",
        },
    ]
    display(pd.DataFrame(rows).set_index("bộ dữ liệu"))

    t_ref, t_min = int(tr["ts"].max()), int(tr["ts"].min())
    x = (t_ref - tr["ts"]) / (t_ref - t_min)  # Δt_norm of Stage 1 (T_ref = max train ts)
    q = x.quantile([0.25, 0.5, 0.75, 0.9])
    say(
        f"- **Δt_norm của {len(tr):,}".replace(",", ".") + f" cạnh train**: trung vị {num(q[0.5], 3)}; tứ phân vị "
        f"{num(q[0.25])} và {num(q[0.75])}; phân vị 90 là {num(q[0.9])}. "
        f"{pct(((t_ref - tr['ts']) <= 30 * 86400).mean())} cạnh nằm trong 30 ngày cuối, "
        f"{pct(((t_ref - tr['ts']) <= 365 * 86400).mean())} trong 365 ngày cuối. "
        f"Cửa sổ train {num((t_ref - t_min) / 86400 / 365.25, 2)} năm (báo cáo dữ liệu: "
        f"{V('lastfm-u25.dt.window_years')} năm)."
    )
    say(
        "**Đọc:** khác MovieLens (rating nhập theo phiên nên Δt theo user gần bằng 0), thời gian của Last-FM trải "
        "rộng ở cả phía user và phía item. "
        'Stage 1 được đo trong điều kiện thuận lợi hơn, nên kết quả "không có tác dụng" ở phần 4 không thể giải '
        'thích bằng "Δt không có thông tin".'
    )
else:
    say("*chưa có: cần `data/processed/lastfm-u25/` (script 02).*")

**Dấu vết của hạn chế quan trọng nhất của dữ liệu: số cạnh train theo tuần.** Tập cặp của bản Last-FM mà KGAT công bố chỉ chứa các cặp còn được
nghe từ 01/04/2014. Nếu đúng, số cặp mới mỗi tuần phải nhảy lên quanh mốc đó (đường đỏ):

In [ ]:
if HAVE_DATA:
    weekly = eda.weekly_counts(tr)
    weekly = weekly[weekly.index >= "2013-10-01"]
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.bar(range(len(weekly)), weekly.values, width=0.8)
    ax.set_xticks(range(0, len(weekly), 4))
    ax.set_xticklabels([d.strftime("%d/%m/%y") for d in weekly.index[::4]], rotation=45, ha="right")
    ax.set_ylabel("cạnh train theo tuần (lần nghe sớm nhất)")
    marker = pd.Timestamp("2014-03-31")  # Monday of the week that contains 2014-04-01
    if marker in weekly.index:
        pos = list(weekly.index).index(marker)
        ax.axvline(pos - 0.5, color="red", ls="--")
        ax.text(pos - 0.3, weekly.max() * 0.92, "01/04/2014", color="red")
    ax.set_title(f"{DS}: số cạnh train theo tuần")
    plt.tight_layout()
    plt.show()
    say(
        f"Trung bình {V(f'{DS}.week.before_mean')} cạnh mỗi tuần trong 4 tuần trước 31/03/2014 và "
        f"{V(f'{DS}.week.after_mean')} trong 4 tuần từ 31/03/2014 "
        "(`docs/data-report-lastfm-numbers.md`, tính từ `train.csv` bằng `eda.weekly_counts`)."
    )

Phần Pair selection của quyết định 0012, nguyên văn (nguồn của hạn chế này, và con số kiểm trên mẫu user):

In [ ]:
m = re.search(
    r"  - Pair selection:(.*?)\n  - The test set",
    read_text("docs/decisions/0012-lastfm-dataset.md"),
    re.S,
)
say("> " + " ".join(m.group(1).split()) if m else "*chưa có: không tìm thấy đoạn Pair selection trong 0012.*")
say(
    '**Hệ quả:** tập cặp được chọn bằng thông tin sau mốc, có thể thiên vị các cặp "sống lâu"; `check_split` không '
    "thấy được lỗi này vì nó nằm trước khi chia. "
    "Tác động lên kết quả **chưa đo** (phương án A, tập cặp không giới hạn bởi cửa sổ tháng 4/2014, chưa chạy lại "
    "trên `lastfm-u25`: quyết định 0013, mục 7)."
)

**KG theo thời gian.** Quy tắc như MovieLens (quyết định 0004 và 0006): KG tại mốc *t* chỉ giữ fact đã đúng trước *t*; KG cho train/val dựng tại t1,
cho test dựng tại t2. KG Last-FM không có ngày của statement: ngày của fact là ngày tồn tại của đích (sớm nhất của `release_date`,
`active_start`, `date_of_birth`).

In [ ]:
k = "lastfm-u25.kg."
say(
    f"""
- **Từ KG gốc tới KG của pipeline:** {V(k + "author_triples")} triple → {V(k + "item_seeded")} (lọc theo item) →
  {V(k + "head_seeded")} (chở được trong định dạng 1-hop; bỏ {V(k + "dropped_not_1hop")} =
  {V(k + "dropped_not_1hop_share")}, head không phải item của train).
  {V(k + "entities")} thực thể ({V(k + "tail_entities")} thực thể đích), {V(k + "relations")} quan hệ.
- **Fact không có ngày:** {V(k + "undated_t1")} triple ({V(k + "undated_t1_share")});
  {V(k + "relations_fully_undated")} quan hệ hoàn toàn không có ngày.
- **Bộ lọc thời gian gần như không cắt gì:** chỉ {V(k + "filtered_t1")} triple ({V(k + "filtered_t1_share")}) bị lọc
  tại t1 (MovieLens: {M("ml100k.filtered_share_t1")} ở ML-100K, {M("ml1m.filtered_share_t1")} ở ML-1M).
  KG tại t1 và t2 giống hệt nhau ({V(k + "default.t1")} và {V(k + "default.t2")} triple); số thực thể chỉ có ở t2:
  {V(k + "entities_only_in_t2")}.
  **Last-FM không đo được hiện tượng "fact tương lai rơi giữa hai mốc"** (ví dụ *Titanic*); câu hỏi rò rỉ qua KG vẫn
  trả lời bằng MovieLens.
"""
)
variants = pd.DataFrame(
    {
        "KG tại t1": [
            V(k + "default.t1"),
            V(k + "drop_undated.t1"),
            V(k + "static.t1"),
        ],
        "KG tại t2": [
            V(k + "default.t2"),
            V(k + "drop_undated.t2"),
            V(k + "static.t2"),
        ],
    },
    index=["default", "drop_undated", "static (tham chiếu)"],
)
display(variants)
rel = sorted(
    ((key.removeprefix(k + "no_date."), NUMS[key][0]) for key in NUMS if key.startswith(k + "no_date.")),
    key=lambda kv: -float(kv[1].rstrip("%").replace(",", ".")),
)
say("**Tỷ lệ fact không có ngày theo quan hệ** (`lastfm-u25`):")
display(pd.DataFrame(rel, columns=["quan hệ", "không có ngày"]).set_index("quan hệ"))

**Thang Δt của Stage 2** (quyết định 0009: `dt_norm` của cạnh KG bị chặn ở 1). Đây là điểm Last-FM khác MovieLens rõ nhất:

In [ ]:
rows = [
    {
        "bộ dữ liệu": label,
        "trong cửa sổ (dt_norm < 1)": V(f"{key}.dt.inside_window_share"),
        "bị chặn ở 1": V(f"{key}.dt.capped_share"),
        "không có ngày": V(f"{key}.dt.undated_share"),
        "cửa sổ train (năm)": V(f"{key}.dt.window_years"),
    }
    for key, label in (
        ("lastfm-u25", "Last-FM u25"),
        ("ml-100k", "ML-100K"),
        ("ml-1m-u10", "ml-1m-u10 (10% user của ML-1M)"),
    )
]
display(pd.DataFrame(rows).set_index("bộ dữ liệu"))
say(
    f"Trên `{DS}` {V('lastfm-u25.dt.inside_window_share')} số triple nằm trong cửa sổ, nên Stage 2 có độ phân giải "
    "để so fact cũ với fact mới "
    f"(MovieLens: {V('ml-100k.dt.inside_window_share')} và {V('ml-1m-u10.dt.inside_window_share')}). "
    f"Cảnh báo: {V('lastfm-u25.dt.capped_share')} số triple vẫn bị chặn ở 1 (trung vị `dt_norm` của fact có ngày: "
    f"{V('lastfm-u25.dt.median_dated')}), "
    "nên kết luận chỉ áp dụng cho phần fact đủ mới. Ngày của fact là ngày tồn tại của đích, không phải lúc fact vào KG."
)

**Bằng chứng không rò rỉ.** Sáu kiểm tra của `check_split` chạy trên split `lastfm-u25` đã lưu; sau đó cố tình chèn một dòng test vào train và
một dòng test vào mask, **chỉ trong bộ nhớ**, để thấy kiểm tra bắt được:

In [ ]:
def saved_split(config_file):
    cfg = load_config(ROOT / "configs/data" / config_file)
    cfg["processed_dir"] = str(ROOT / cfg["processed_dir"])
    return load_saved(cfg)[0]


if HAVE_DATA:
    s = saved_split("lastfm_u25.yaml")
    say(f"- `{DS}`: `check_split` báo **{len(check_split(s))}** vấn đề.")
    broken = {
        "chèn một dòng test vào train": dataclasses.replace(
            s, train=pd.concat([s.train, s.test.head(1)], ignore_index=True)
        ),
        "chèn một dòng test vào mask của val": dataclasses.replace(
            s,
            seen_before_val=pd.concat(
                [s.seen_before_val, s.test.head(1)[["user", "item", "ts"]]],
                ignore_index=True,
            ),
        ),
    }
    for what, bad in broken.items():
        problems = check_split(bad)
        say(f"- **{what}** ({DS}, trong bộ nhớ): {len(problems)} vấn đề được phát hiện:")
        for p in problems:
            print("   ", p)
else:
    say(
        "*chưa có: cần `data/processed/lastfm-u25` (chạy `scripts/02_build_splits.py --data "
        "configs/data/lastfm_u25.yaml`).*"
    )
checks = re.findall(r"^\s*# (\d)\. (.+)$", read_text("src/tkgat/data/leakage.py"), re.M)
say("**Sáu kiểm tra** (đọc từ `src/tkgat/data/leakage.py`):\n\n" + "\n".join(f"{i}. {text}" for i, text in checks))
say(
    "`check_split` **không** bắt được hai nguồn rò rỉ: chọn mô hình bằng test (chỉ `scripts/run.py` chấm test, sau "
    "khi chọn trên val) "
    "và tri thức tương lai trong KG (`check_kg`). Và nó không thấy được thiên lệch chọn mẫu của tập cặp ở trên."
)

## 3. Huấn luyện mô hình
*Người trình bày: Phôn (baseline, quy trình), Triết (KGAT, T-KGAT) (đề xuất)*

**Quy trình chung** (giống MovieLens): mỗi điểm lưới × 5 seed (2019–2023); Recall@20 trên val mỗi 5 epoch; dừng sớm khi val không tăng
(patience 5 với BPR-MF, 10 với KGAT; tối đa 300 epoch); giữ epoch tốt nhất trên val; chọn điểm lưới có trung bình Recall@20 val cao nhất;
**chấm test một lần**; mọi dòng val và test ghi vào `results/registry.csv` (chỉ thêm, có git commit). T-KGAT giữ (lr, l2) của KGAT
và chỉ dò γ ∈ {1, 3, 10}.

**Tham số huấn luyện BPR-MF** (`docs/bprmf-training-table.md`, cột `lastfm-u25`):

In [ ]:
lines = [ln for ln in read_text("docs/bprmf-training-table.md").splitlines() if ln.startswith("|")]
cells = [[c.strip() for c in ln.strip().strip("|").split("|")] for ln in lines]
display(pd.DataFrame([r[:2] for r in cells[2:]], columns=cells[0][:2]).set_index(cells[0][0]))
figure("fig5_bprmf_curve_lastfm-u25.png")
say(
    "Hình: loss huấn luyện và Recall@20 val của BPR-MF (trung bình ± std qua 5 seed); vạch chấm là epoch tốt nhất "
    "của từng seed, **chọn theo val**, test không tham gia."
)

**Lưới và điểm được chọn** (trung bình Recall@20 **val** qua 5 seed, từ registry; điểm in đậm trong bài nói là điểm có val cao nhất của từng mô hình):

In [ ]:
VAL = REG[REG["phase"] == "val"].copy()
DROP = {"dt_den", "t_ref", "perm_seed", "l2_kg", "mess_dropout"}


def short(cfg):
    d = json.loads(cfg)
    return ", ".join(f"{k} = {v}" for k, v in d.items() if k not in DROP)


VAL["điểm lưới"] = VAL["config"].map(short)
g = (
    VAL[VAL["model"].isin(["bprmf", "kgat", "kgat_nokg", "tkgat", "tkgat_kg"])]
    .groupby(["model", "điểm lưới"])
    .agg(val_recall20=("recall@20", "mean"), n_seed=("seed", "nunique"))
)
g["val_recall20"] = g["val_recall20"].map(num)
display(g)


def epoch_range(model, which):
    # the best epochs of one model at its selected grid point, as docs/data-report-lastfm-numbers.md wrote them
    return V(f"{DS}.best_epoch.{model}.chosen.{which}")


def epoch_cap(model):
    # the epoch limit of the model's own config (kgat.yaml for the KGAT family, bprmf.yaml for BPR-MF)
    return int(V("conv.max_epochs_bprmf" if model == "bprmf" else "conv.max_epochs"))


def as_int(text):
    return int(text.replace(".", ""))


train_models = [m for m in sorted(VAL["model"].unique()) if not m.startswith("mostpop")]  # MostPop has no epochs
epochs = pd.DataFrame(
    [
        {
            "mô hình": m,
            "epoch tốt nhất, nhỏ nhất": epoch_range(m, "min"),
            "epoch tốt nhất, lớn nhất": epoch_range(m, "max"),
            "giới hạn epoch": epoch_cap(m),
        }
        for m in train_models
    ]
).set_index("mô hình")
say(
    "**Epoch tốt nhất chọn trên val** của điểm lưới đã chọn (trung bình Recall@20 val cao nhất; "
    f"nhỏ nhất và lớn nhất qua {V('conv.seeds')} seed, từ `docs/data-report-lastfm-numbers.md`, mục 6):"
)
display(epochs)
at_cap = [m for m in train_models if as_int(epoch_range(m, "max")) >= epoch_cap(m)]
if at_cap:
    say(
        "Ở điểm đã chọn, ít nhất một seed của "
        + ", ".join(f"`{m}`" for m in at_cap)
        + " có epoch tốt nhất bằng giới hạn epoch: huấn luyện của seed đó dừng vì hết giới hạn, chưa dừng sớm trên val."
    )
else:
    say("Ở điểm đã chọn, không seed nào của mô hình nào có epoch tốt nhất bằng giới hạn epoch.")
say(
    "Điểm được chọn ở các mô hình nằm ở **mép lưới** (ví dụ BPR-MF l2 lớn nhất, KGAT lr lớn nhất, γ = 10 ở Stage 1 "
    "và γ = 1 ở Stage 2), và chênh lệch val giữa các γ nhỏ "
    "(xem bảng): đây là hạn chế đã biết, chưa mở rộng lưới."
)

**Learning curve val của KGAT, T-KGAT và bản hoán vị** (nét liền: đủ các seed; nét đứt: còn ít seed hơn). Ba đường chồng lên nhau trong dải ± 1 độ lệch chuẩn qua seed, và các đường còn tăng chậm ở nửa sau của trục epoch (đọc từ hình, không phải số đã tính), nên không đọc được khác biệt giữa ba mô hình từ hình này:

In [ ]:
figure("rq3_curves_lastfm-u25.png", 720)

**Tính toán và Kaggle** (quyết định 0013, 0014). Thời gian chạy lấy từ cột `seconds` của các dòng val trong registry, nhóm theo thiết bị (`docs/data-report-lastfm-numbers.md`, mục 6):

In [ ]:
dev_prefix = f"{DS}.hours.device."
slugs = [k.removeprefix(dev_prefix) for k in NUMS if k.startswith(dev_prefix)]
hours = pd.Series(
    {s.replace("_", " "): V(dev_prefix + s) for s in slugs},
    name="giờ (tổng cột seconds, dòng val)",
)
display(hours.to_frame())
by_model = pd.DataFrame(
    [
        {
            "thiết bị": s.replace("_", " "),
            "mô hình": k.removeprefix(f"{DS}.hours.{s}."),
            "giờ": NUMS[k][0],
        }
        for s in slugs
        for k in NUMS
        if k.startswith(f"{DS}.hours.{s}.")
    ]
).set_index(["thiết bị", "mô hình"])
display(by_model)

d14 = read_text("docs/decisions/0014-lastfm-stage2-kaggle.md")
gate = re.search(r"  2\. Gate:(.*?)\n  3\. ", d14, re.S)
outcome = re.search(r"- Outcome(.*?)\n- Consequences:", d14, re.S)
say(
    "**Quyết định 0014, mục 2 (cổng đo), nguyên văn:**\n\n> "
    + (" ".join(gate.group(1).split()) if gate else "*chưa có*")
)
say(
    "**Quyết định 0014, Outcome (kết quả thực tế), nguyên văn:**\n\n> "
    + (" ".join(outcome.group(1).split()) if outcome else "*chưa có*")
)
say(
    "**Hệ quả cho cách đọc:** Stage 2 chạy trên Kaggle (Tesla T4), KGAT và Stage 1 trên laptop (RTX 3050); so Stage "
    "2 với KGAT hoặc Stage 1 là so giữa hai máy, "
    "so `tkgat_kg` với `tkgat_kg_perm` là cùng một máy."
)

## 4. Kết quả và nhận xét
*Người trình bày: Phôn (baseline), Triết (KGAT, T-KGAT) (đề xuất)*

> **Kết quả sơ bộ, chưa đóng băng (tag `v0.2-kgat`), chạy trước khi cô duyệt giao thức chia theo mốc.** `lastfm-u25` là 25% user của Last-FM, không phải Last-FM đầy đủ.

Bảng, kiểm định và hình do `scripts/10_rq3_table.py` và `scripts/09_significance.py` ghi (`results/rq3_models.csv`, `results/rq3_pairs.csv`, `results/significance.csv`);
notebook chỉ đọc chúng. Recall@20 và NDCG@20 trên **test**, trung bình ± độ lệch chuẩn mẫu (ddof = 1) qua 5 seed:

In [ ]:
PRELIMINARY = {"kgat", "kgat_nokg", "tkgat", "tkgat_perm", "tkgat_kg", "tkgat_kg_perm"}
models = pd.read_csv(ROOT / "results/rq3_models.csv")
models = models[models["dataset"] == DS]
pairs = pd.read_csv(ROOT / "results/rq3_pairs.csv")
pairs = pairs[pairs["dataset"] == DS]


def sample_std(row, metric):
    return None if pd.isna(row[f"{metric}_std"]) else row[f"{metric}_std"]


rows = []
for _, r in models.iterrows():
    ok = r["status"] == "ok"
    rows.append(
        {
            "mô hình": r["label"] + (" [sơ bộ]" if r["model"] in PRELIMINARY else ""),
            "điểm chọn trên val": r["point"] if ok else "—",
            "Recall@20 (test)": vn_pm(r["recall@20_mean"], sample_std(r, "recall@20")) if ok else NOT_RUN,
            "NDCG@20 (test)": vn_pm(r["ndcg@20_mean"], sample_std(r, "ndcg@20")) if ok else NOT_RUN,
            "số seed": int(r["n_seeds"]) if ok else "—",
        }
    )
say(f"**{DS}** (user test: {int(models['n_users'].iloc[0])})")
show(pd.DataFrame(rows))

**Kiểm định cặp theo user** (Wilcoxon, trung bình các seed chung; hiệu = trước − sau; p Holm trong cùng bộ dữ liệu và metric; `n_nonzero` là cỡ mẫu hiệu dụng):

In [ ]:
rows = []
for _, r in pairs.iterrows():
    ok = r["status"] == "ok"
    note = r.get("warning") if ok else r["status"]
    rows.append(
        {
            "so sánh": r["pair"],
            "metric": r["metric"],
            "n_pairs": int(r["n_pairs"]) if ok else "—",
            "n_nonzero": int(r["n_nonzero"]) if ok else "—",
            "hiệu trung bình": num(r["mean_diff"]) if ok else "—",
            "p thô": vn_p(r["p_raw"]) if ok else "—",
            "p Holm": vn_p(r["p_holm"]) if ok else "—",
            "ghi chú": note if isinstance(note, str) else "(không cảnh báo)",
        }
    )
show(pd.DataFrame(rows))

**Baseline** (`results/significance.csv`; hiệu = mô hình A − mô hình B; MostPop và MostPop recent chỉ có một lần chạy):

In [ ]:
sig = pd.read_csv(ROOT / "results/significance.csv")
sig = sig[sig["dataset"] == DS]
cols = [
    "metric",
    "model_a",
    "model_b",
    "n_seeds_a",
    "n_pairs",
    "n_nonzero",
    "mean_diff",
    "p_raw",
    "p_holm",
]
show(
    sig[cols]
    .fillna("")
    .assign(
        mean_diff=lambda d: d["mean_diff"].map(lambda x: num(x) if x != "" else x),
        p_raw=lambda d: d["p_raw"].map(lambda x: vn_p(x) if x != "" else x),
        p_holm=lambda d: d["p_holm"].map(lambda x: vn_p(x) if x != "" else x),
    )
)

**Nhận xét** (các số trong lời đọc được tính từ đúng các bảng trên):

In [ ]:
mm = models.set_index("model")


def mean(model, metric="recall@20"):
    return mm.loc[model, f"{metric}_mean"]


def std(model, metric="recall@20"):
    return mm.loc[model, f"{metric}_std"]


def holm(a, b, metric):
    r = pairs[(pairs["model_a"] == a) & (pairs["model_b"] == b) & (pairs["metric"] == metric)].iloc[0]
    return vn_p(r["p_holm"]), num(r["mean_diff"])


def base(a, b, metric):
    r = sig[(sig["model_a"] == a) & (sig["model_b"] == b) & (sig["metric"] == metric)].iloc[0]
    return vn_p(r["p_holm"])


n_users = int(models["n_users"].iloc[0])
sel_window = models.set_index("model").loc["mostpop_recent", "point"]
p_tk, d_tk = holm("tkgat", "kgat", "recall@20")
p_tk_n, d_tk_n = holm("tkgat", "kgat", "ndcg@20")
p_tp, d_tp = holm("tkgat", "tkgat_perm", "recall@20")
p_tp_n, d_tp_n = holm("tkgat", "tkgat_perm", "ndcg@20")
p_kn, d_kn = holm("kgat", "kgat_nokg", "recall@20")
p_kn_n, d_kn_n = holm("kgat", "kgat_nokg", "ndcg@20")
p_s2, d_s2 = holm("tkgat_kg", "kgat", "recall@20")
p_s2_n, _ = holm("tkgat_kg", "kgat", "ndcg@20")
p_s2p, d_s2p = holm("tkgat_kg", "tkgat_kg_perm", "recall@20")
p_s2p_n, _ = holm("tkgat_kg", "tkgat_kg_perm", "ndcg@20")
nz = pairs["n_nonzero"]
say(
    f"""
**1. Baseline.** BPR-MF đạt Recall@20 {num(mean("bprmf"))} so với MostPop {num(mean("mostpop"))}, và thắng cả MostPop
lẫn MostPop recent rất có ý nghĩa
(p Holm {base("bprmf", "mostpop", "recall@20")} và {base("bprmf", "mostpop_recent", "recall@20")}). **MostPop recent**
(cửa sổ chọn trên val: {sel_window}) đạt {num(mean("mostpop_recent"))},
**thấp hơn** MostPop thường (NDCG@20: p Holm {base("mostpop", "mostpop_recent", "ndcg@20")}). Giải thích chỉ là giả
thuyết, chưa kiểm:
cửa sổ neo ở cuối train, còn test bắt đầu ở t2, sau t1 khá lâu.

**2. KG (RQ2).** KGAT {num(mean("kgat"))} ± {num(std("kgat"))} so với KGAT không KG {num(mean("kgat_nokg"))} ±
{num(std("kgat_nokg"))}: hiệu {d_kn} Recall@20 và {d_kn_n} NDCG@20,
**không có ý nghĩa** (p Holm {p_kn} và {p_kn_n}). KGAT cũng chỉ hơn BPR-MF ({num(mean("bprmf"))} ±
{num(std("bprmf"))}) {num(mean("kgat") - mean("bprmf"))}, nhỏ hơn độ lệch chuẩn qua seed của BPR-MF;
cặp này **chưa nằm trong họ kiểm định** nên chưa có p. **Chưa có bằng chứng KG giúp trên Last-FM.**

**3. Thời gian, Stage 1 (RQ3).** T-KGAT {num(mean("tkgat"))} ± {num(std("tkgat"))}; KGAT {num(mean("kgat"))}; bản hoán
vị {num(mean("tkgat_perm"))} ± {num(std("tkgat_perm"))}.
T-KGAT so với KGAT: hiệu {d_tk} / {d_tk_n}, p Holm {p_tk} / {p_tk_n}. T-KGAT thật so với hoán vị: hiệu {d_tp} /
{d_tp_n}, p Holm {p_tp} / {p_tp_n}.
**Chưa có bằng chứng thời gian trên cạnh user–item giúp**, và bản hoán vị không kém bản thật.

**4. Thời gian, Stage 2.** Stage 2 {num(mean("tkgat_kg"))} ± {num(std("tkgat_kg"))}; bản hoán vị ngày fact
{num(mean("tkgat_kg_perm"))} ± {num(std("tkgat_kg_perm"))}.
So với KGAT: hiệu {d_s2}, p Holm {p_s2} / {p_s2_n}; so với bản hoán vị: hiệu {d_s2p}, p Holm {p_s2p} / {p_s2p_n}.
Thang Δt của KG ở đây có độ phân giải (phần 2) nên đây là phép đo thật
cho "fact cũ hay mới", dù phần lớn fact có ngày vẫn bị chặn ở 1. Lưu ý so sánh với KGAT là giữa hai máy.

**Mức độ chắc chắn.** Test có {n_users} user và `n_nonzero` từ {int(nz.min())} đến {int(nz.max())}, không có cảnh báo
cỡ mẫu: kiểm định đủ sức để nói chênh lệch (nếu có) nhỏ,
nhỏ hơn độ nhiễu qua seed. Đây vẫn **không** phải bằng chứng "bằng nhau".
"""
)

## 5. Minh bạch và hạn chế
*Người trình bày: Triết (đề xuất)*

In [ ]:
if HAVE_DATA:
    t2 = int(meta["t2"])
    val_days = (t2 - t1) / 86400
    test_days = (int(pd.read_csv(ROOT / f"data/processed/{DS}/test.csv")["ts"].max()) - t2) / 86400
    span = (
        f"train {num((t_ref - t_min) / 86400, 0)} ngày, val {num(val_days, 0)} ngày, test {num(test_days, 0)} "
        "ngày (tính từ dữ liệu đã xử lý)"
    )
else:
    span = "(chưa có dữ liệu đã xử lý)"
say(
    f"""
**Phải báo rõ:**
1. Quyết định chuyển sang Last-FM được đưa ra **sau khi** điểm test MovieLens đã được xem (0012, phần Context ở trên).
2. KGAT và T-KGAT là **sơ bộ**: bảng RQ3 ghi "chạy trước khi cô duyệt giao thức"; chưa có tag `v0.2-kgat`; PR #42 còn
  là bản nháp và chưa tái lập độc lập từ clone sạch (tình trạng ngày 12/11/2026, cập nhật tay nếu đã đổi).
3. Stage 2 chạy trên máy khác (Kaggle T4); cổng đo thời gian và ước lượng giờ chạy: xem 0014 ở phần 3, nguyên văn.
4. Các phân tích bổ sung đã đề xuất với cô (0014, Consequences) **chưa chạy**; mỗi cái cần quyết định riêng và sẽ được
  ghi là thêm sau khi đã thấy test.

**Hạn chế:**
1. Tập cặp (user, item) của KGAT được chọn bằng thông tin sau 01/04/2014; ảnh hưởng chưa đo (phần 2).
2. Chỉ {V("lastfm-u25.user_fraction")} user; mất {V("lastfm-u25.val_dropped_cold_item_share")} val và
  {V("lastfm-u25.test_dropped_cold_item_share")} test vì cold item; không nói cho toàn Last-FM.
3. Timestamp là lần nghe sớm nhất của cặp; độ dài khoảng thời gian rất lệch nhau: {span}.
4. Freebase là bản chụp {V("conv.freebase_snapshot")}; KG không có ngày của statement;
  {V("lastfm-u25.kg.undated_t1_share")} fact không có ngày; KG t1 = KG t2 nên không kiểm được rò rỉ giữa hai mốc.
5. KGAT không tái lập từng bit trên GPU; chưa tái lập độc lập.
6. Điểm được chọn nằm ở mép lưới; hoán vị toàn cục chỉ là một dạng đối chứng (hoán vị theo khối thời gian chưa chạy).
7. Stage 2 so với KGAT là so giữa hai máy; BPR-MF so với KGAT chưa kiểm định.
8. Số của nhóm không so trực tiếp với số trong paper KGAT (chia theo thời gian, 5-core, 25% user, KG có ngày).
"""
)

## 6. Kế hoạch và các câu hỏi xin ý kiến cô
*Người trình bày: Cả hai (đề xuất)*

**Kế hoạch (đề xuất, cần cô chọn):** hoàn tất tái lập độc lập và đóng băng kết quả (tag `v0.2-kgat`); xử lý hạn chế của tập cặp; chọn một số phân tích bổ sung theo ý cô; sau đó viết khóa luận.

**Ba câu hỏi xin ý kiến cô:**
1. **Tập cặp của KGAT bị chọn bằng thông tin sau 01/04/2014.** Cô có muốn nhóm dựng lại tập cặp từ sự kiện thô của LFM-1b, không giới hạn bởi cửa sổ tháng 4/2014 (phương án A, quyết định 0013 mục 7),
   và chạy lại trên `lastfm-u25` không? Việc này cần quyết định mới và thêm thời gian GPU.
2. **Các phân tích bổ sung đã đề xuất** (hoán vị theo khối thời gian, nhóm từ 01/04/2014, biến thể `drop_undated`, γ ∈ {30, 100} chỉ trên val, MostPop và BPR-MF trên Last-FM đầy đủ):
   cô ưu tiên cái nào? Mỗi cái có quyết định riêng và được báo là thêm sau khi đã thấy test.
3. **Kết quả sơ bộ:** giữ các số `lastfm-u25` hiện tại làm chính thức sau khi tái lập độc lập và gắn tag, hay cô muốn chỉnh giao thức rồi chạy lại?